# Exploratory data analysis: historical financial figures and prices

This notebook inspects the source workbook, investigates the MGS price anomaly, and applies the project's existing cleaning functions. The raw DataFrames remain available for comparison. Run the cells in order from the project root or the `notebooks` directory.


## 1. Setup and Data Loading

Load both worksheets from the project workbook. Daily dates are parsed as in the original notebook; annual availability dates remain in their source form until the cleaning step. No files are written.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

workbook = project_root / "data" / "Raw" / "Historical_Data.xlsx"
if not workbook.is_file():
    raise FileNotFoundError(f"Source workbook not found: {workbook}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_cleaner import clean_annual_figures, clean_daily_prices
from src.validation import validate_daily_prices

annual_figures = pd.read_excel(workbook, sheet_name="Annual figures")
prices = pd.read_excel(workbook, sheet_name="Daily prices", parse_dates=["Date"])


## 2. Dataset Overview

`annual_figures` contains company and year financial observations, including fundamentals and sector specific fields. `prices` contains daily closing prices for the company tickers and a benchmark. These raw DataFrames are retained throughout the notebook.


In [ ]:
print(f"Annual figures: {annual_figures.shape[0]:,} rows × {annual_figures.shape[1]} columns")
print(f"Daily prices: {prices.shape[0]:,} rows × {prices.shape[1]} columns")
display(annual_figures.head(), prices.head())


In [ ]:
print("Annual figures columns and types:")
display(annual_figures.dtypes.rename("dtype").to_frame())
print("Daily prices columns and types:")
display(prices.dtypes.rename("dtype").to_frame())


## 3. Data Quality Assessment

Inspect missingness, record and identifier duplicates, date coverage, and price validity before cleaning. Sector specific annual fields may be absent by design; missing values are reported rather than filled.


In [ ]:
print("Missing values in annual figures:")
display(annual_figures.isna().sum().rename("missing_count").to_frame())
print("Missing values in daily prices:")
display(prices.isna().sum().rename("missing_count").to_frame())

print("Duplicate annual rows:", annual_figures.duplicated().sum())
print("Duplicate company-year identifiers:", annual_figures.duplicated(["ticker", "year"]).sum())
print("Duplicate daily rows:", prices.duplicated().sum())
print("Duplicate trading dates:", prices["Date"].duplicated().sum())


In [ ]:
annual_dates = pd.to_datetime(annual_figures["available_date"], errors="coerce")
price_dates = pd.to_datetime(prices["Date"], errors="coerce")

print("Annual fiscal years:", annual_figures["year"].min(), "to", annual_figures["year"].max())
print("Annual availability dates:", annual_dates.min(), "to", annual_dates.max())
print("Trading dates:", price_dates.min(), "to", price_dates.max())
print("Invalid annual availability dates:", annual_dates.isna().sum())
print("Invalid trading dates:", price_dates.isna().sum())


The next checks identify non positive prices and extreme one day price changes. Negative financial flows or profits are not automatically invalid, so this notebook does not modify them.


In [ ]:
price_columns = prices.columns.drop("Date")
print("Non positive price counts by series:")
display((prices[price_columns] <= 0).sum().rename("non_positive_count").to_frame())
print("Non positive share counts:", (annual_figures["shares_m"] <= 0).sum())

daily_returns = prices.sort_values("Date").set_index("Date")[price_columns].pct_change()
print("Largest absolute daily price returns:")
display(daily_returns.abs().stack().nlargest(10).rename("absolute_return").to_frame())


## 4. MGS Price Anomaly Investigation

The raw MGS closing price on **17 June 2024** is **1755.506900**, while neighbouring observations are around 17. Dividing that observation by 100 gives the hypothesised value **17.555069**. A GBX to GBP unit mismatch is a plausible explanation, but the workbook does not verify the cause. The raw June chart below preserves the original notebook's investigation.


In [ ]:
june_2024 = prices.loc[
    prices["Date"].between("2024-06-01", "2024-06-30")
].sort_values("Date")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    june_2024["Date"],
    june_2024["MGS"],
    marker="o",
    markersize=3,
    label="MGS close (raw)",
)
ax.set(
    title="Marigold Systems daily closing price, June 2024",
    xlabel="Date",
    ylabel="Recorded closing price (source units)",
)
ax.grid(alpha=0.3)
ax.legend()
fig.autofmt_xdate()
plt.show()


Compare the recorded and hypothesised values without changing `prices`. Percentage returns use the same chronological MGS observations on both sides of 17 June; the only scenario change is dividing that day's price by 100.


In [ ]:
anomaly_date = pd.Timestamp("2024-06-17")
raw_mgs = prices.sort_values("Date").set_index("Date")["MGS"]
hypothesised_mgs = raw_mgs.copy()
hypothesised_mgs.loc[anomaly_date] /= 100

mgs_comparison = pd.DataFrame({
    "recorded_price": raw_mgs,
    "hypothesised_price": hypothesised_mgs,
    "recorded_return": raw_mgs.pct_change(),
    "hypothesised_return": hypothesised_mgs.pct_change(),
})
display(mgs_comparison.loc["2024-06-13":"2024-06-20"])
print("Recorded anomaly price:", f"{raw_mgs.loc[anomaly_date]:.6f}")
print("Hypothesised price:", f"{hypothesised_mgs.loc[anomaly_date]:.6f}")


In [ ]:
june_comparison = mgs_comparison.loc["2024-06-01":"2024-06-30"]
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes[0].plot(june_comparison.index, june_comparison["recorded_price"], marker="o", label="Recorded")
axes[0].plot(june_comparison.index, june_comparison["hypothesised_price"], marker="o", label="17 June ÷ 100")
axes[0].set(title="MGS price comparison, June 2024", ylabel="Recorded price (source units)")
axes[1].plot(june_comparison.index, june_comparison["recorded_return"], marker="o", label="Recorded")
axes[1].plot(june_comparison.index, june_comparison["hypothesised_return"], marker="o", label="17 June ÷ 100")
axes[1].set(title="Effect on daily returns", xlabel="Date", ylabel="Daily return (decimal)")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 5. Data Cleaning

Apply the current project functions to copies of the raw data. The annual cleaner converts availability dates, rejects duplicate company year identifiers, and sorts records; it preserves sector specific missing values. The daily cleaner converts and sorts dates, rejects duplicate dates and non positive prices, then divides the 17 June 2024 MGS value by 100. This correction is an existing project rule, despite the unit explanation remaining unverified.


In [ ]:
cleaned_annual_figures = clean_annual_figures(annual_figures)
cleaned_prices = clean_daily_prices(prices)


## 6. Post-Cleaning Validation

These checks report remaining missing values and invalid identifiers, dates, or prices. They do not modify either cleaned DataFrame.


In [ ]:
print("Daily price validation:")
display(pd.Series(validate_daily_prices(cleaned_prices), name="result").to_frame())
print("Annual missing values:")
display(cleaned_annual_figures.isna().sum().rename("missing_count").to_frame())
print("Duplicate company-year identifiers:", cleaned_annual_figures.duplicated(["ticker", "year"]).sum())
print("Invalid annual availability dates:", cleaned_annual_figures["available_date"].isna().sum())
print("Non positive cleaned prices:", (cleaned_prices.drop(columns="Date") <= 0).sum().sum())


In [ ]:
assert cleaned_annual_figures.shape == annual_figures.shape
assert cleaned_prices.shape == prices.shape
cleaned_mgs_price = cleaned_prices.loc[
    cleaned_prices["Date"].eq(anomaly_date), "MGS"
].iloc[0]
assert abs(cleaned_mgs_price - raw_mgs.loc[anomaly_date] / 100) < 1e-10

print("Cleaned annual shape:", cleaned_annual_figures.shape)
print("Cleaned daily shape:", cleaned_prices.shape)
print("Cleaned MGS price on 17 June 2024:", f"{cleaned_mgs_price:.6f}")


## 7. Summary and Findings

- The source contains **200 annual observations** and **2,529 daily observations**. No duplicate records, invalid dates, or non positive daily prices were found in this workbook.
- Annual sector specific fields contain missing values; the annual cleaner preserves them. No missing values were found in the daily price sheet.
- The recorded MGS price on 17 June 2024 is **1755.506900**. The existing cleaner applies the **÷ 100** correction, producing **17.555069**; a GBX to GBP unit mismatch remains an unverified hypothesis.
- The correction changes the MGS daily returns on 17 and 18 June 2024 from approximately **+9,998.5%** and **−99.01%** to **+0.985%** and **−0.958%** respectively.
- Cleaning converts date fields, sorts records, checks duplicate identifiers and non positive prices, and applies the documented MGS correction. The cleaned DataFrames, `cleaned_annual_figures` and `cleaned_prices`, remain **in memory only** and have **not been exported**.
- Before modelling or backtesting, confirm the MGS source unit with the data provider and decide how to handle sector specific missing fields for each analysis.
